In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
llm = ChatOpenAI(model="gpt-4o-mini")

llm.invoke([HumanMessage("잘 지냈어?")])

AIMessage(content='저는 잘 지내고 있습니다! 당신은 어떻게 지내고 계신가요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 12, 'total_tokens': 31, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f9565ce549', 'id': 'chatcmpl-DWcQQgVdR76MWr89UsZV63uB8Cbof', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da994-791e-7111-aed6-4bc1acc24471-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 19, 'total_tokens': 31, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [2]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool
def get_current_time(timezone: str, location: str) -> str:
    """
    현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존(예: 'Asia/Seoul'). 실제 존재햐애함
        location (str): 지역명. 타임존은 모든 지명에 대응되지 않으므로 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재 시각 {now}'
    print(location_and_local_time)
    return location_and_local_time

In [3]:
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

llm_with_tools = llm.bind_tools(tools)

In [4]:
from langchain_core.messages import SystemMessage


messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?")
]

response = llm_with_tools.invoke(messages)
messages.append(response)

print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 132, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f957560a82', 'id': 'chatcmpl-DWcaktRjmIV1FAhzDts2L97MQIQSM', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da99e-3e88-76f3-af65-9d7f4165ae9a-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_67Oa4MzZObtpAvF1zuJxia9I', 'type': 'tool_call'}],

In [5]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재 시각 2026-04-20 15:42:19


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 132, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f957560a82', 'id': 'chatcmpl-DWcaktRjmIV1FAhzDts2L97MQIQSM', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da99e-3e88-76f3-af65-9d7f4165ae9a-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_67Oa4MzZObtpAvF1zuJxia9I', 'type': 'tool_call'}

In [6]:
llm_with_tools.invoke(messages)

AIMessage(content='부산은 현재 2026년 4월 20일 오후 3시 42분입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 188, 'total_tokens': 212, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f957560a82', 'id': 'chatcmpl-DWcd9MgY8H6ZOMa3YwsBcTh71xby7', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da9a0-8256-7d51-9c92-0c2fd9a7daa6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 188, 'output_tokens': 24, 'total_tokens': 212, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [7]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")

In [8]:
import yfinance as yf


@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown()

    return history_md

tools = [get_current_time, get_yf_stock_history,]
tool_dict = {
    "get_current_time" : get_current_time,
    "get_yf_stock_history" : get_yf_stock_history,
}

llm_with_tools = llm.bind_tools(tools)

In [9]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐어? 내렸어?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 280, 'total_tokens': 307, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_42fcdba006', 'id': 'chatcmpl-DWcvGIuUrmbeSgJemkxTrs1uzfVGu', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019da9b1-a832-7dc2-bdb3-755996e8954b-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}, 'id': 'call_clZh9GxE6nSeA5ObXkhKiRl5', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 280, 'output_tokens': 27, 'total_tokens': 307, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audi

In [12]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)

{'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}
content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2026-03-18 00:00:00-04:00 | 399    | 403.07 | 392.31 |  392.78 | 5.08531e+07 |           0 |              0 |\n| 2026-03-19 00:00:00-04:00 | 387.27 | 387.27 | 378.73 |  380.3  | 6.70783e+07 |           0 |              0 |\n| 2026-03-20 00:00:00-04:00 | 379.85 | 379.89 | 364.46 |  367.96 | 7.86286e+07 |           0 |              0 |\n| 2026-03-23 00:00:00-04:00 | 373.09 | 385.33 | 372.73 |  380.85 | 7.4606e+07  |           0 |              0 |\n| 2026-03-24 00:00:00-04:00 | 376.56 | 387.48 | 376.31 |  383.03 | 6.00049e+07 |           0 |              0 |\n| 2026-03-25 00:00:00-04:00 | 389.99 | 396.23 | 385.01 |  385.95 | 5.51573e+07 |           0 |              0 |\n| 2026-03-26 00:00:00-04:0

In [13]:
llm_with_tools.invoke(messages)

AIMessage(content='테슬라(TSLA)의 주가 변화를 살펴보면, 한 달 전인 2026년 3월 18일 주가가 392.78 달러였고, 현재 2026년 4월 17일 주가는 400.62 달러입니다. \n\n따라서, 테슬라의 주가는 한 달 사이에 올랐습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 1637, 'total_tokens': 1721, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_42fcdba006', 'id': 'chatcmpl-DWd5KWwhXJfie0pNQsB2rSN9udAE8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da9bb-2c9b-7870-b29c-e027ba709add-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1637, 'output_tokens': 84, 'total_tokens': 1721, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0